In [ ]:
import os
from pyspark.sql import SparkSession
import subprocess
import time

# Kill process di port 15002
subprocess.run('lsof -ti:15002 | xargs -r kill -9', shell=True)
time.sleep(1)

# Stop session sebelumnya biar port 15002 lepas
try:
    spark.stop()
    print("🛑 Spark session sebelumnya dihentikan")
except:
    pass

def create_spark_session(env="local"):
    """
    Spark Session adaptif untuk:
    - 'local'  : MinIO (development/local testing)
    - 'aws'    : AWS S3 (production via Terraform)
    """
    # Package AWS untuk Hadoop 3.3.x
    aws_packages = "org.apache.hadoop:hadoop-aws:3.4.2,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.postgresql:postgresql:42.7.3"
    builder = SparkSession.builder \
        .appName("Franchise-Data-Pipeline") \
        .config("spark.driver.memory", "2g") \
        .config("spark.jars.packages", aws_packages) \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.skewJoin.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .config("spark.hadoop.fs.s3a.multipart.size", "104857600") \
        .config("spark.hadoop.fs.s3a.threads.max", "20") \
        .config("spark.hadoop.fs.s3a.committer.name", "magic") \
        .config("spark.plugins", "org.apache.spark.sql.connect.SparkConnectPlugin") \
        .config("spark.sql.connect.server.port", "0")  # <-- FIXED: Pakai 0 biar auto-assign
    
    # Kustomisasi per environment
    if env == "local":
        print("🔧 LOCAL: Connecting to MinIO at localhost:9000")
        builder = builder \
            .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
            .config("spark.hadoop.fs.s3a.path.style.access", "true") \
            .config("spark.hadoop.fs.s3a.aws.credentials.provider", 
                    "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
            .config("spark.hadoop.fs.s3a.access.key", os.getenv("MINIO_ACCESS_KEY", "minioadmin")) \
            .config("spark.hadoop.fs.s3a.secret.key", os.getenv("MINIO_SECRET_KEY", "minioadmin"))
            
    elif env == "aws":
        print("☁️ AWS: Connecting to S3 with IAM Role (via Terraform)")
        builder = builder \
            .config("spark.hadoop.fs.s3a.path.style.access", "false") \
            .config("spark.hadoop.fs.s3a.aws.credentials.provider", 
                    "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
            .config("spark.hadoop.fs.s3a.endpoint", "")
    
    return builder.getOrCreate()

# =========================================================================
# PENGGUNAAN
# =========================================================================
RUNNING_ENV = "aws"
spark = create_spark_session(env=RUNNING_ENV)

print("✅ Spark session nyala")
print(f"Spark Version: {spark.version}")
print(f"Spark Master: {spark.sparkContext.master}")

☁️ AWS: Connecting to S3 with IAM Role (via Terraform)


26/05/28 06:33:00 WARN SparkContext: Another SparkContext is being constructed (or threw an exception in its constructor). This may indicate an error, since only one SparkContext should be running in this JVM (see SPARK-2243). The other SparkContext was created at:
org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:59)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:77)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:500)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:481)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.command

✅ Spark session nyala
Spark Version: 4.1.1
Spark Master: local[*]


26/05/28 06:34:14 WARN CredentialProviderListFactory: Credentials option fs.s3a.aws.credentials.provider contains AWS v1 SDK entry com.amazonaws.auth.DefaultAWSCredentialsProviderChain
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/05/28 11:09:51 WARN ExecuteGrpcResponseSender: Got detached from opId=59e93d38-975c-4708-9a7a-268582bbd2ff at index 4.totalTime=5012.460903 ms waitingForResults=5007.716338 ms waitingForSend=0.0 ms
26/05/28 11:09:51 ERROR FileFormatWriter: Aborting job 42d56ee1-5335-42ff-b248-cc50aee3cfac.
java.io.InterruptedIOException: getFileStatus on s3a://franchise-pipeline-dev-data-lake-silver/outlet_master/year=2026/month=02/day=25: software.amazon.awssdk.core.exception.AbortedException: Thread was interrupted
	at org.apache.hadoop.fs.s3a.S3AUtils.translateInterruptedException(S3AUtils.java:468)
	at org.a

In [4]:
# Test koneksi
try:
    if RUNNING_ENV == "local":
        spark.read.csv("s3a://franchise-pipeline-data-lake-bronze/order_items/year=2026/month=03/day=01/order_items.csv").show(5)
    else:
        spark.read.csv("s3a://franchise-pipeline-dev-data-lake-bronze/order_items/year=2026/month=03/day=02/order_items.csv").show(5)
    print("✅ Connection successful!")
except Exception as e:
    print(f"❌ Connection failed: {e}")

+-------+--------+-------+--------+--------------+--------+
|    _c0|     _c1|    _c2|     _c3|           _c4|     _c5|
+-------+--------+-------+--------+--------------+--------+
|item_id|order_id|menu_id|quantity|price_per_item|subtotal|
|2517633| 1045592|      3|       1|      22000.00|22000.00|
|2482817| 1031165|     11|       2|      39000.00|78000.00|
|2482818| 1031165|     20|       2|      25000.00|50000.00|
|2482819| 1031165|     30|       3|      26000.00|78000.00|
+-------+--------+-------+--------+--------------+--------+
only showing top 5 rows
✅ Connection successful!
